In [1]:
import os
import re
from collections.abc import Callable

import pandas as pd

from utils import PROCESSED_DIR, RESULTS_DIR, clean_styler

# Directory containing raw observations
DATA_DIR = "data/original"

# Overwrite outputs
OVERWRITE = True

# Fu2023's assumed mass (µg) per particle
PARTICLE_MASS = {"land": 0.057, "ocean": 0.1}


def add_metadata(obs: pd.DataFrame, metadata: dict) -> pd.DataFrame:
    """Add metadata"""

    # Add metadata identifying source study
    for study, info in metadata.items():
        idx = info.pop("idx")
        try:
            author, year = re.split(r"(\d{4})", study)[0:2]
            year = int(year)
        except ValueError:
            author = study
            year = None
        obs.loc[idx, "author"] = author
        obs.loc[idx, "year"] = year
        for col, value in info.items():
            obs.loc[idx, col] = value

    return obs


def annotate_fu2023(data_dir: str = DATA_DIR) -> pd.DataFrame:
    """Load Fu2023 observations and add metadata"""

    conc = annotate_fu2023_conc(data_dir=data_dir)
    depo = annotate_fu2023_depo(data_dir=data_dir)
    obs = pd.concat(
        [conc.assign(measure="concentration"), depo.assign(measure="deposition")]
    ).reset_index(drop=True)

    # fmt: off
    columns = [
        "author", "year", "doi", "measure", "lat", "lon", "particle/m3", "ug/m3",
        "particle/m2/d", "t/km2/yr", "size_min_um", "size_max_um", "shape", "area",
        "notes"
    ]
    # fmt: on

    return obs[columns]


def annotate_fu2023_conc(data_dir: str = DATA_DIR) -> pd.DataFrame:
    """Load observed concentration and add metadata"""

    # Define metadata
    metadata = {
        "Allen2021": {
            "idx": 0,
            "doi": "https://doi.org/10.1038/s41467-021-27454-7",
            "size_min_um": 3.5,
            "size_max_um": 53,
            "shape": "Fibers + Fragments",
            "area": "land",
            "notes": "size range from results section",
        },
        "Liu2019a": {
            "idx": range(1, 4),
            "doi": "https://doi.org/10.1021/acs.est.9b03427",
            "size_min_um": 16,
            "size_max_um": 2087,
            "shape": "Fibers + Fragments",
            "area": "ocean",
            "notes": "size range from results section",
        },
        "Abbasi2019": {
            "idx": 4,
            "doi": "https://doi.org/10.1016/j.envpol.2018.10.039",
            "size_min_um": 10,
            "size_max_um": 500,
            "shape": "Fibers",
            "area": "land",
            "notes": "size range estimated from section 3.3",
        },
        "Unknown-c1": {"idx": 5, "area": "land", "notes": "North of Bushehr, Iran"},
        "Allen2020": {
            "idx": 6,
            "doi": "https://doi.org/10.1371/journal.pone.0232746",
            "size_min_um": 5,
            "size_max_um": 140,
            "shape": "Not reported",
            "area": "land",
            "notes": "size range from results section",
        },
        "Gaston2020": {
            "idx": 7,
            "doi": "https://doi.org/10.1177/0003702820920652",
            "size_min_um": 25,
            "size_max_um": 2061,
            "shape": "Fibers + Fragments",
            "area": "land",
            "notes": "size range from table 2",
        },
        "Dris2017": {
            "idx": 8,
            "doi": "https://doi.org/10.1016/j.envpol.2016.12.013",
            "size_min_um": 50,
            "size_max_um": 1650,
            "shape": "Fibers",
            "area": "land",
            "notes": "size range from figure 2",
        },
        "Liu2019b": {
            "idx": 9,
            "doi": "https://doi.org/10.1016/j.scitotenv.2019.04.110",
            "size_min_um": 23,
            "size_max_um": 9555,
            "shape": "Fibers + Fragments",
            "area": "land",
            "notes": "size range from section 3.2",
        },
        "Liu2020": {
            "idx": range(10, 19),
            "doi": "https://doi.org/10.1016/j.jhazmat.2020.123223",
            "size_min_um": 16,
            "size_max_um": 2986,
            "shape": "Fibers",
            "area": "land",
            "notes": "size range from section 3.3",
        },
        "Yuan2023": {
            "idx": 19,
            "doi": "https://doi.org/10.1016/j.scitotenv.2023.161839",
            "size_min_um": 13,
            "size_max_um": 334,
            "shape": "Fibers",
            "area": "land",
            "notes": "size range from section 3.2",
        },
        "Ding2021": {
            "idx": 20,
            "doi": "https://doi.org/10.1016/j.atmosenv.2021.118389",
            "size_min_um": 50,
            "size_max_um": 2210,
            "shape": "Fibers + Films + Foams + Fragments",
            "area": "ocean",
            "notes": "size range from section 3.2",
        },
        "Wang2021": {
            "idx": 21,
            "doi": "https://doi.org/10.1016/j.jhazmat.2021.125477",
            "size_min_um": 19,
            "size_max_um": 949,
            "shape": "Fibers + Films + Fragments",
            "area": "ocean",
            "notes": "size range from section 3.2",
        },
        "Wang2020": {
            "idx": range(22, 43),
            "doi": "https://doi.org/10.1016/j.jhazmat.2019.121846",
            "size_min_um": 59,
            "size_max_um": 2252,
            "shape": "Fibers + Fragments",
            "area": "ocean",
            "notes": "size range from section 3.2",
        },
        "Trainic2020": {
            "idx": range(43, 52),
            "doi": "https://doi.org/10.1038/s43247-020-00061-y",
            "size_min_um": 17,
            "size_max_um": 172,
            "shape": "Not reported",
            "area": "ocean",
            "notes": "size range from table 1",
        },
        "Caracci2023": {
            "idx": range(52, 72),
            "doi": "https://doi.org/10.1016/j.jhazmat.2023.131036",
            "size_min_um": 1,
            "size_max_um": 50,
            "shape": "Fibers + Fragments",
            "area": "ocean",
            "notes": (
                "size range estimated from methods (sampling with PM10 inlet head may"
                + " capture fibers longer than 10 um)"
            ),
        },
    }

    # Load and set metadata
    path = os.path.join(data_dir, "fu2023_concentration.txt")
    obs = load_fu2023(path)

    return add_metadata(obs=obs, metadata=metadata)


def annotate_fu2023_depo(data_dir: str) -> pd.DataFrame:
    """Load observed deposition and add metadata"""

    # Define metadata
    metadata = {
        # Do Liu2020 first to set dtypes for all new columns
        "Liu2020": {
            "idx": range(8, 12),
            "doi": "https://doi.org/10.1016/j.jhazmat.2020.123223",
            "size_min_um": 16,
            "size_max_um": 4900,
            "shape": "Fibers",
            "area": "land",
            "notes": "size range estimated from figure 3b",
        },
        "Unknown-d1": {"idx": 0, "area": "ocean", "notes": "Bohai Sea"},
        "Unknown-d2": {"idx": 1, "area": "ocean", "notes": "Bohai Sea"},
        "Unknown-d3": {"idx": 2, "area": "land", "notes": "Eastern Shandong, China"},
        "Unknown-d4": {"idx": 3, "area": "land", "notes": "Tianjin, China"},
        "Unknown-d5": {"idx": 4, "area": "land", "notes": "West of Dalian, China"},
        "Unknown-d6": {"idx": 5, "area": "land", "notes": "Eastern Shandong, China"},
        "Unknown-d7": {"idx": 6, "area": "land", "notes": "East of Dongguan, China"},
        "Unknown-d8": {"idx": 7, "area": "land", "notes": "Tibet, China"},
        "Cai2017": {
            "idx": 12,
            "doi": "https://doi.org/10.1007/s11356-017-0116-x",
            "size_min_um": 20,
            "size_max_um": 5000,
            "shape": "Fibers + Films + Foams + Fragments",
            "area": "land",
            "notes": "size range estimated from figure 4",
        },
        "Tian2020": {
            "idx": range(13, 16),
            "doi": "https://doi.org/10.13671/j.hjkxxb.2020.0067",
            "area": "land",
            "notes": "could not access article",
        },
        "Brahney2020": {
            "idx": range(16, 27),
            "doi": "https://doi.org/10.1126/science.aaz5819",
            "size_min_um": 4,
            "size_max_um": 3000,
            "shape": "Fibers + Fragments",
            "area": "land",
            "notes": "size range stated in paper",
        },
        "Allen2019": {
            "idx": 27,
            "doi": "https://doi.org/10.1038/s41561-019-0335-5",
            "size_min_um": 10,
            "size_max_um": 3000,
            "shape": "Fibers + Films + Fragments",
            "area": "land",
            "notes": "size range estimated from figure 2",
        },
        "Klein2019": {
            "idx": range(28, 31),
            "doi": "https://doi.org/10.1016/j.scitotenv.2019.05.405",
            "size_min_um": 20,
            "size_max_um": 5000,
            "shape": "Fibers + Fragments",
            "area": "land",
            "notes": "size range estimated from section 3.3",
        },
        "Roblin2020": {
            "idx": 31,
            "doi": "https://doi.org/10.1021/acs.est.0c04000",
            "size_min_um": 50,
            "size_max_um": 4000,
            "shape": "Fibers",
            "area": "land",
            "notes": (
                "size range estimated from figure S3 and statement that visual detection"
                + " was limited to fibers with length > 50 um"
            ),
        },
        "Dris2016": {
            "idx": 32,
            "doi": "https://doi.org/10.1016/j.marpolbul.2016.01.006",
            "size_min_um": 50,
            "size_max_um": 3200,
            "shape": "Fibers",
            "area": "land",
            "notes": "size range estimated from figure 3",
        },
        "Stanton2019": {
            "idx": 33,
            "doi": "https://doi.org/10.1016/j.scitotenv.2019.02.278",
            "shape": "Fibers",
            "area": "land",
            "notes": "size range not reported",
        },
        "Wright2020": {
            "idx": [34, 35],
            "doi": "https://doi.org/10.1016/j.envint.2019.105411",
            "size_min_um": 25,
            "size_max_um": 2900,
            "shape": "Fibers + Films + Foams + Fragments",
            "area": "land",
            "notes": "size range estimated from figure 2",
        },
        "Yuan2023": {
            "idx": 36,
            "doi": "https://doi.org/10.1016/j.scitotenv.2023.161839",
            "size_min_um": 29,
            "size_max_um": 185,
            "shape": "Fibers",
            "area": "land",
            "notes": "size range from section 3.2",
        },
    }

    # Load and set metadata
    path = os.path.join(data_dir, "fu2023_deposition.txt")
    obs = load_fu2023(path)

    return add_metadata(obs=obs, metadata=metadata)


def load_fu2023(path: str) -> pd.DataFrame:
    """Load Fu2023's observations"""

    # Define column names; read units from second row
    number_units, mass_units = pd.read_table(path, header=1, nrows=0).columns[-2:]
    number_units = number_units.replace("MPs/", "particle/")
    mass_units = mass_units.replace("/a", "/yr")
    cols = ["type", "year", "mon", "lon", "lat", number_units, mass_units]

    # Load data file
    return pd.read_table(path, header=1, names=cols)


def reformat_columns(
    obs: pd.DataFrame, number_units: str, mass_units: str
) -> pd.DataFrame:
    """Reformat number and mass columns, storing units as data entries"""

    obs = obs.rename(columns={number_units: "number", mass_units: "mass"})
    obs.insert(obs.columns.get_loc("number") + 1, "number_units", number_units)
    obs.insert(obs.columns.get_loc("mass") + 1, "mass_units", mass_units)

    return obs


def revise_fu2023(obs: pd.DataFrame) -> pd.DataFrame:
    """Revise Fu2023 observations and merge into single table"""

    conc = obs.loc[obs["measure"].eq("concentration"), :].copy()
    conc = revise_fu2023_conc(obs=conc)
    conc = reformat_columns(conc, number_units="particle/m3", mass_units="ug/m3").drop(
        columns=["particle/m2/d", "t/km2/yr"]
    )

    depo = obs.loc[obs["measure"].eq("deposition"), :].copy()
    depo = revise_fu2023_depo(obs=depo)
    depo = reformat_columns(
        depo, number_units="particle/m2/d", mass_units="t/km2/yr"
    ).drop(columns=["particle/m3", "ug/m3"])

    obs = (
        pd.concat([conc, depo])
        .reset_index(drop=True)
        .rename(columns=lambda x: x.removesuffix("_um"))
    )
    obs.insert(obs.columns.get_loc("size_max") + 1, "size_units", "µm")

    return obs


def revise_fu2023_conc(obs: pd.DataFrame) -> pd.DataFrame:
    """Fix inconsistencies in original data"""

    def update(doi: str, updates: dict) -> None:
        """Update data and possibly recompute mass concentration"""

        mask = obs["doi"].fillna("Unknown").str.endswith(doi)

        if "particle/m3" in updates:
            area = obs.loc[mask, "area"].unique()
            if len(area) > 1:
                raise ValueError(f"Rows to update must have same area; got {area}")
            area = area[0]

            ug_per_particle = PARTICLE_MASS[area]
            updates["ug/m3"] = round(ug_per_particle * updates["particle/m3"], 15)

        if "notes" in updates:
            updates["notes"] = (
                obs.loc[mask, "notes"].astype(str) + "; " + updates["notes"]
            )

        for k, v in updates.items():
            obs.loc[mask, k] = v

    # Abbasi2019
    # Fix lat/lon (was mislocated); fix number (increase precision)
    update(
        doi="10.1016/j.envpol.2018.10.039",
        updates={"lat": 27.47, "lon": 52.61, "particle/m3": 0.657},
    )

    # Allen2021
    # Fix lat/lon (was mislocated); fix number (increase precision)
    update(
        doi="10.1038/s41467-021-27454-7",
        updates={"lat": 42.94, "lon": 0.14, "particle/m3": 0.233},
    )

    # Caracci2023
    # Recompute number (was rounded) to match mass (measured by study)
    doi = "10.1016/j.jhazmat.2023.131036"
    mask = obs["doi"].fillna("Unknown").str.endswith(doi)
    area = obs.loc[mask, "area"].unique()  # sanity check
    if len(area) > 1:
        raise ValueError(f"Rows to recompute must have same area; got {area}")
    area = area[0]
    ug_per_particle = PARTICLE_MASS[area]
    obs.loc[mask, "particle/m3"] = (obs.loc[mask, "ug/m3"] / ug_per_particle).round(15)
    update(
        doi=doi,
        updates={
            "notes": "counts estimated from study-reported mass assuming 100 ng/particle"
        },
    )

    # Ding2021
    # Fix number (was rounded)
    update(doi="10.1016/j.atmosenv.2021.118389", updates={"particle/m3": 0.035})

    # Dris2017
    # Fix lat/lon (was mislocated)
    update(doi="10.1016/j.envpol.2016.12.013", updates={"lat": 48.8, "lon": 2.5})

    # Gaston2020
    # Fix lat/lon (was mislocated); fix number (use count of plastic particles identified
    # via Nile red fluorescence rather than via particle colour)
    update(
        doi="10.1177/0003702820920652",
        updates={
            "lat": 34.17,
            "lon": -118.96,
            "particle/m3": 6.2,
            "notes": "count is particles identified via Nile red fluorescence",
        },
    )

    # Liu2019b
    # Fix lat/lon (was mislocated)
    update(doi="10.1016/j.scitotenv.2019.04.110", updates={"lat": 31.1, "lon": 121.5})

    # Trainic2020
    # Round lat/lon to remove false precision
    doi = "10.1038/s43247-020-00061-y"
    mask = obs["doi"].fillna("Unknown").str.endswith(doi)
    update(
        doi=doi,
        updates={
            "lat": obs.loc[mask, "lat"].round(5),
            "lon": obs.loc[mask, "lon"].round(5),
        },
    )

    # Wang2021
    # Fix number (use mean of all samples)
    update(
        doi="10.1016/j.jhazmat.2021.125477",
        updates={"particle/m3": 0.0039, "notes": "count is mean of all samples"},
    )

    # Drop rows with zero microplastics
    obs = obs.loc[obs["particle/m3"].gt(0)].reset_index(drop=True)

    # Ensure mass = number * average mass per particle
    for area in ["land", "ocean"]:
        mask = obs["area"].eq(area)
        obs.loc[mask, "ug/m3"] = (
            obs.loc[mask, "particle/m3"] * PARTICLE_MASS[area]
        ).round(15)

    return obs.sort_values(["author", "year", "doi"])


def revise_fu2023_depo(obs: pd.DataFrame) -> pd.DataFrame:
    """Fix inconsistencies in original data"""

    CONVERSION = 1e-12 * 1e6 * 365  # t/ug * m2/km2 * days/year

    def update(doi: str, updates: dict) -> None:
        """Update data and possibly recompute mass deposition"""

        mask = obs["doi"].fillna("Unknown").str.endswith(doi)

        if "particle/m2/d" in updates:
            area = obs.loc[mask, "area"].unique()
            if len(area) > 1:
                raise ValueError(f"Rows to update must have same area; got {area}")
            area = area[0]

            ug_per_particle = PARTICLE_MASS[area]
            updates["t/km2/yr"] = round(
                updates["particle/m2/d"] * CONVERSION * ug_per_particle, 15
            )

        if "notes" in updates:
            updates["notes"] = (
                obs.loc[mask, "notes"].astype(str) + "; " + updates["notes"]
            )

        for k, v in updates.items():
            obs.loc[mask, k] = v

    # Allen2019
    # Fix lat/lon (was mislocated)
    update(doi="10.1038/s41561-019-0335-5", updates={"lat": 42.804, "lon": 1.419})

    # Brahney2020
    # Fix lat/lon (were mislocated; new values from NADP station data)
    # fmt: off
    lats = [
        36.0586, 42.9290, 43.4605, 40.2878, 34.0695, 40.7543, 38.4584, 40.0547, 38.9561,
        39.0054, 37.6186
    ]
    lons = [
        -112.1840, -109.7875, -113.5551, -105.6628, -116.3889, -109.4671, -109.8210,
        -105.5891, -106.9860, -114.2170, -112.1728,
    ]
    # fmt: on
    update(doi="10.1126/science.aaz5819", updates={"lat": lats, "lon": lons})

    # Cai2017
    # Fix number (exclude organic particles)
    update(
        doi="10.1007/s11356-017-0116-x",
        updates={
            "particle/m2/d": 36,
            "notes": "count excludes organic particles e.g. cellulose",
        },
    )

    # Dris2016
    # Fix lat/lon (were mislocated)
    update(doi="10.1016/j.marpolbul.2016.01.006", updates={"lat": 48.805, "lon": 2.443})

    # Klein2019
    # Fix lat/lon (mean of all six sample locations); fix number (mean of all samples)
    update(
        doi="10.1016/j.scitotenv.2019.05.405",
        updates={
            "lat": 53.518,
            "lon": 9.923,
            "particle/m2/d": 305.1,
            "notes": "count is mean of all samples",
        },
    )

    # Roblin2020
    # Fix lat/lon (location of bulk deposition sample); fix number (value of bulk
    # deposition sample excluding organic fibers)
    update(
        doi="10.1021/acs.est.0c04000",
        updates={
            "lat": 55.37175,
            "lon": -7.33945,
            "particle/m2/d": 15,
            "notes": (
                "count from bulk deposition sample only"
                + "; count excludes organic fibers e.g. cellulose"
            ),
        },
    )

    # Wright2020
    # Fix lat/lon; fix number (sum of fibers and fragments)
    update(
        doi="10.1016/j.envint.2019.105411",
        updates={"lat": 51.5111, "lon": -0.1171, "particle/m2/d": 771},
    )

    # Unknown studies: adjust number to match mass and round
    # Since we cannot verify the counts, we update them to match the reported masses to
    # minimize the change in the masses when we recompute them from the counts using the
    # assumed mass per particle
    for area in ["land", "ocean"]:
        mask = obs["author"].str.startswith("Unknown") & obs["area"].eq(area)
        obs.loc[mask, "particle/m2/d"] = (
            obs.loc[mask, "t/km2/yr"] / CONVERSION / PARTICLE_MASS[area]
        ).round(1)

    # Drop duplicate rows (Klein2019, Wright2020)
    obs = obs.drop_duplicates().reset_index(drop=True)

    # Drop rows with zero microplastics
    obs = obs.loc[obs["particle/m2/d"].gt(0)].reset_index(drop=True)

    # Ensure mass = number * unit conversion * average mass per particle
    for area in ["land", "ocean"]:
        mask = obs["area"].eq(area)
        obs.loc[mask, "t/km2/yr"] = (
            obs.loc[mask, "particle/m2/d"] * CONVERSION * PARTICLE_MASS[area]
        ).round(15)

    return obs.sort_values(["author", "year", "doi"])


def save_or_load(
    out_name: str,
    fun: Callable,
    out_dir: str = PROCESSED_DIR,
    overwrite: bool = False,
    **kwargs,
) -> pd.DataFrame:
    """Wrapper to skip processing if output file already exists"""

    out_path = os.path.join(out_dir, out_name)
    if os.path.exists(out_path) and not overwrite:
        print(f"File exists: {out_path}")
        return pd.read_csv(out_path)

    result: pd.DataFrame = fun(**kwargs)
    result.to_csv(out_path, index=False)
    print(f"-> {out_path}")

    return result


Introduction
------------

This notebook processes observations of atmospheric microplastic concentration and deposition so that we can use them to constrain our simulations.

The observations were collected from the literature by @Fu2023, who used them to constrain simulations of atmospheric microplastic cycling. We obtained the data from Yanxu Zhang.

Annotate @Fu2023's observations
-------------------------------

@Fu2023's observational dataset does not identify which study each observation was sourced from. We manually match the obserations to studies in the literature (when possible) based on location and microplastic particle count and add metadata taken from the study text:

* Source study
* Observed particle size range
* Observed microplastic shape(s)

In [2]:
fu2023 = save_or_load(
    out_name="obs_fu2023.csv", fun=annotate_fu2023, data_dir=DATA_DIR, overwrite=OVERWRITE
)

-> data/processed/obs_fu2023.csv


In [3]:
# Print summary for reference
clean_styler(
    fu2023.groupby(["author", "year", "doi", "measure", "area"], dropna=False)
    .agg(
        {
            "lat": "count",
            "size_min_um": "min",
            "size_max_um": "max",
            "particle/m3": "mean",
            "ug/m3": "mean",
            "particle/m2/d": "mean",
            "t/km2/yr": "mean",
        }
    )
    .sort_index(level="measure")
    .rename(columns={"lat": "n_obs"})
    .style.set_caption("Microplastic observations compiled by Fu et al. (2023)")
    .format(precision=1)
    .format(precision=3, subset=["particle/m3", "particle/m2/d"])
    .format(precision=4, subset=["ug/m3", "t/km2/yr"])
    .format_index(precision=0)
)

Revise @Fu2023's observations
-----------------------------

In some cases the data of @Fu2023 are inconsistent with the study from which they are taken. We update the data to ensure that:

* Lat/lon matches study-reported location
* Number concentration/deposition matches study-reported value

We also recompute mass from number using @Fu2023's assumed average particle mass of 57 ng/particle over land and 100 ng/particle over oceans. We do this because the number and mass values do not always correspond, likely due to rounding. For example, if the data show a number concentration over land of 1 particle/m^3^ and mass concentration of 0.1 µg/m^3^ we change the mass concentration to 0.057.

For the concentration observations sourced from @Caracci2023, which reported microplastic particle mass rather than particle counts, we recompute number from mass assuming 100 ng/particle (all samples were collected over the Atlantic).

We drop any observations without microplastics (concentration or deposition = 0).

In [4]:
fu2023_revised = save_or_load(
    out_name="obs_fu2023-revised.csv",
    fun=revise_fu2023,
    out_dir=RESULTS_DIR,
    overwrite=OVERWRITE,
    obs=fu2023,
)

-> results/obs_fu2023-revised.csv


In [5]:
# Print summary for reference(
clean_styler(
    fu2023_revised.groupby(["author", "year", "doi", "measure", "area"], dropna=False)
    .agg(
        {
            "lat": "count",
            "size_min": "min",
            "size_max": "max",
            "size_units": "first",
            "number": "mean",
            "number_units": "first",
            "mass": "mean",
            "mass_units": "first",
        }
    )
    .sort_index(level="measure")
    .rename(columns={"lat": "n_obs"})
    .style.set_caption(
        "Revised observations of microplastic concentration based on Fu et al. (2023)"
    )
    .format(precision=1)
    .format(precision=3, subset=["number"])
    .format(precision=4, subset=["mass"])
    .format_index(precision=0)
)